In [12]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights


class ViTForCifar10(nn.Module):
    """
    A wrapper for the ViT model, modified for CIFAR-10.
    This version assumes input tensors are ALREADY preprocessed.
    """
    def __init__(self, num_classes=10, in_channels=3, image_size=32, patch_size=4):
        super().__init__()
        
        # --- 1. Load Pre-trained ViT Model ---
        self.vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        
        # --- 2. Update All Necessary Internal Model Attributes ---
        
        # a) THE DEFINITIVE FIX: Update the patch_size attribute 🔧
        self.vit.patch_size = patch_size
        
        # b) Update the image size attribute
        self.vit.image_size = image_size
        
        # c) Calculate the new number of patches and sequence length
        num_patches = (image_size // patch_size) ** 2
        seq_length = num_patches + 1
        
        # d) Update the encoder's sequence length attribute
        self.vit.encoder.seq_length = seq_length
        
        # --- 3. Modify the Model Architecture ---
        hidden_dim = self.vit.hidden_dim

        # a) Modify the Patch Embedding layer to match the new patch size
        self.vit.conv_proj = nn.Conv2d(
            in_channels=in_channels,
            out_channels=hidden_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

        # b) Modify the Positional Embeddings to match the new sequence length
        self.vit.encoder.pos_embedding = nn.Parameter(
            torch.randn(1, seq_length, hidden_dim)
        )

        # c) Modify the Classifier Head for the new number of classes
        self.vit.heads.head = nn.Linear(
            in_features=hidden_dim,
            out_features=num_classes
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs a forward pass on a preprocessed tensor.
        
        Args:
            x (torch.Tensor): A preprocessed tensor of shape (B, 3, 32, 32).
        
        Returns:
            torch.Tensor: Logits of shape (B, num_classes).
        """
        return self.vit(x)

# --- Main execution block to demonstrate and verify the model ---
if __name__ == "__main__":
    
    # 1. Instantiate the fully corrected model
    model = ViTForCifar10(num_classes=NUM_CLASSES)
    model.eval()

    # 2. Create a dummy input tensor with the exact required shape
    batch_size = 4
    dummy_preprocessed_tensor = torch.randn(batch_size, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
    
    print("\n--- Verifying Model with a Preprocessed Dummy Tensor ---")
    print(f"Shape of dummy tensor: {dummy_preprocessed_tensor.shape}")

    # 3. Perform a forward pass
    with torch.no_grad():
        try:
            output = model(dummy_preprocessed_tensor)
            print(f"Forward pass successful! ✅")
            print(f"Shape of model output: {output.shape}")
            
            # Verify the output shape is correct
            assert output.shape == (batch_size, NUM_CLASSES)
            print("\nThe model now runs correctly without any shape errors.")
        except Exception as e:
            print(f"\nForward pass failed: {e}")

Model initialized and all internal attributes are now correctly updated. ✅

--- Verifying Model with a Preprocessed Dummy Tensor ---
Shape of dummy tensor: torch.Size([4, 3, 32, 32])
Forward pass successful! ✅
Shape of model output: torch.Size([4, 10])

The model now runs correctly without any shape errors.


In [ ]:
import torch

W = torch.randn(384, 384)
P = torch.randn(384, 2, requires_grad=True)
Q = torch.randn(2, 384, requires_grad=True)

print(torch.linalg.norm(W - P @ Q))

U, S, V = torch.svd_lowrank(W, 2, niter=20)

Vt = V.t()
sqrt_S = torch.sqrt(S)
P.data = U * sqrt_S.unsqueeze(0)        # (out_features, rank)
Q.data = sqrt_S.unsqueeze(1) * Vt      # (rank, in_features)


print(torch.linalg.norm(W - P @ Q))

optim = torch.optim.AdamW([P, Q], lr=1e-5, weight_decay=0)
for i in range(100):

    W = W + (0.001 * torch.randn(384, 384))
    # optim.zero_grad()
    loss = torch.linalg.norm(W.detach() - P @ Q)
    # loss.backward()
    # optim.step()
    print(i, loss.item())

tensor(688.1162, grad_fn=<LinalgVectorNormBackward0>)
tensor(380.6939, grad_fn=<LinalgVectorNormBackward0>)
0 380.6925964355469
1 380.6929016113281
2 380.6918029785156
3 380.6921691894531
4 380.6916809082031
5 380.6920166015625
6 380.6923522949219
7 380.6927795410156
8 380.6933288574219
9 380.6942443847656
10 380.6961975097656
11 380.6957702636719
12 380.6957092285156
13 380.69677734375
14 380.6969909667969
15 380.6979064941406
16 380.6978759765625
17 380.6976013183594
18 380.6970520019531
19 380.6964111328125
20 380.6954345703125
21 380.6951599121094
22 380.6945495605469
23 380.69476318359375
24 380.69586181640625
25 380.69659423828125
26 380.69561767578125
27 380.6946716308594
28 380.6935119628906
29 380.6937255859375
30 380.6944580078125
31 380.69476318359375
32 380.6955871582031
33 380.69757080078125
34 380.696044921875
35 380.6974792480469
36 380.69842529296875
37 380.69879150390625
38 380.6966552734375
39 380.69671630859375
40 380.6983337402344
41 380.69775390625
42 380.699310302